# 01 - H1 Polyzentrik

**Der Digitalhaushalt ist nicht in ein oder zwei „Schatten-Digitalministerien" konzentriert, sondern auf sechs Ressorts mit fundamental verschiedenen Aufgaben-Profilen verteilt.**

Essay-Bezug: Akt 1

## These und politischer Anschluss

**Original-These.** Verteidigung, Inneres und Verkehr sind die „Schatten-Digitalministerien" des Bundes, sie geben jeweils mehr für Digitalisierung aus als das offiziell zuständige Haus.

**Präzisierte These nach erster Sichtung.** Der Digitalhaushalt ist polyzentrisch auf sechs Ressorts mit fundamental verschiedenen Aufgaben-Profilen verteilt. Eine zentrale Steuerung durch das Bundesministerium für Digitales und Staatsmodernisierung (BMDS) erfasst strukturell nur einen begrenzten Teil des Volumens.

**Politischer Anschluss.** Die BMDS-Gründung 2025 lebt von der Erwartung, ein Ministerium könne digitale Mittel bündeln und steuern. Ob das realistisch ist, hängt davon ab, wie konzentriert der Digitalhaushalt tatsächlich ist und ob die großen Ressorts überhaupt vergleichbare Aufgaben haben.

**Was wir prüfen.** Erstens den Konzentrationsgrad über die Jahre (Herfindahl-Hirschman-Index (HHI), Top-Anteile). Zweitens die Ranglisten-Verschiebung 2019 → 2024. Drittens die inhaltlichen Profile der größten Ressorts. Viertens, daraus abgeleitet, drei Szenarien für die realistische BMDS-Bündelungsreichweite.

## Methodik dieses Drilldowns

**Daten und Abgrenzung.** `digi_soll_eng`, aggregiert auf Einzelplan-Ebene (Aggregations-Filter, Ressort-Zuschnitte ändern sich zwischen 2019 und 2024, etwa beim BMDV; Aggregation absorbiert das). Konzentration über `hhi()`, Profile über die anteilige Kategorien-Allokation `kategorien_allokation()` (1/n-Aufteilung bei Mehrfachkategorien).

**Pre-Mortem.** Was würde die polyzentrische These töten? Ein HHI klar über 2.500 (starke Konzentration), ein Ressort mit dominantem Anteil deutlich über einem Drittel, oder Top-Ressorts mit nahezu identischen Aufgaben-Profilen, dann wäre Bündelung naheliegend statt politisch konfliktträchtig.

**Erwartung.** Ein moderat fragmentierter HHI, eine knappe Spitze ohne dominantes Ressort, und Profile, die der jeweiligen Sachaufgabe folgen (Verkehr → Infrastruktur, Verteidigung → Bundeswehr-IT, Bildung → Forschung/Kompetenzen).

In [1]:
"""01 - H1: Polyzentrik des Digitalhaushalts.

Drei Drilldown-Stufen:
  1. Konzentrationsmasse (HHI, Top-Anteile) je Jahr
  2. Ranglisten-Veraenderung 2019 vs. 2024
  3. Profile der Top-Ressorts ueber 9 Kategorien (Allokations-Methode)
  4. BMDS-Buendelungs-Szenarien als Volumen-Quantifizierung

Outputs: figures/h1_profile_2024.png, figures/h1_hhi_zeitreihe.png
"""

'01 - H1: Polyzentrik des Digitalhaushalts.\n\nDrei Drilldown-Stufen:\n  1. Konzentrationsmasse (HHI, Top-Anteile) je Jahr\n  2. Ranglisten-Veraenderung 2019 vs. 2024\n  3. Profile der Top-Ressorts ueber 9 Kategorien (Allokations-Methode)\n  4. BMDS-Buendelungs-Szenarien als Volumen-Quantifizierung\n\nOutputs: figures/h1_profile_2024.png, figures/h1_hhi_zeitreihe.png\n'

In [2]:
import sys
from pathlib import Path
sys.path.insert(0, '..')  # repo root, relativ zum notebooks/-Verzeichnis

import pandas as pd
from src.load import load, kategorien_allokation, hhi, KATS, KAT_KURZ

df = load()  # nutzt Repo-Root-Default aus src.load

In [3]:
#| label: tbl-hhi
print("=== Konzentration ueber Einzelplaene (digi_soll_eng) ===\n")
rows = []
for j in sorted(df['jahr'].unique()):
    ep = df[df['jahr'] == j].groupby('einzelplan')['digi_soll_eng'].sum()
    ep = ep[ep > 0].sort_values(ascending=False)
    total = ep.sum()
    shares = ep / total
    n_80 = (shares.cumsum() < 0.8).sum() + 1
    rows.append({
        'jahr': j,
        'aktive_EP': len(ep),
        'HHI': round(hhi(df, j)),
        'Top1_%': round(shares.iloc[0] * 100, 1),
        'Top3_%': round(shares.iloc[:3].sum() * 100, 1),
        'Top5_%': round(shares.iloc[:5].sum() * 100, 1),
        'EP_fuer_80%': n_80,
        'Sigma_Mrd': round(total / 1e6, 2),
    })
konz = pd.DataFrame(rows)
print(konz.to_string(index=False))

=== Konzentration ueber Einzelplaene (digi_soll_eng) ===

 jahr  aktive_EP  HHI  Top1_%  Top3_%  Top5_%  EP_fuer_80%  Sigma_Mrd
 2019         22 1701    23.1    64.2    85.7            5       8.51
 2021         23 1369    21.1    52.0    76.2            6      16.55
 2023         24 1247    16.0    44.6    70.1            6      19.19
 2024         24 1612    23.0    62.0    84.2            5      17.94


**Befund.** Der HHI bewegt sich in allen vier Jahren in einer Bandbreite, die im international etablierten Vergleich als moderat fragmentiert gilt, weit unter dem Wert eines konzentrierten Marktes (über 2.500) und in der Größenordnung etwa des Bundestags nach Sitzanteilen (rund 1.900). Der Anteil des größten Ressorts bleibt durchweg unter einem Viertel, fünf bis sechs Ressorts werden für 80 Prozent des Volumens gebraucht.

Nächster Schritt: Welche Ressorts stehen 2024 oben — und wie hat sich die Rangfolge gegenüber 2019 verschoben?

In [4]:
print("\n=== Top-10 Ressorts 2019 vs. 2024 (Mrd. EUR) ===")
for j in [2019, 2024]:
    print(f"\n{j}:")
    ep = df[df['jahr'] == j].groupby(
        ['einzelplan', 'einzelplantext']
    )['digi_soll_eng'].sum().div(1e6)
    ep = ep[ep > 0.05].sort_values(ascending=False).head(10)
    for i, ((nr, txt), v) in enumerate(ep.items(), 1):
        print(f"  {i:>2}. EP{nr:>2}  {v:>5.2f} Mrd  {str(txt)[:55]}")


=== Top-10 Ressorts 2019 vs. 2024 (Mrd. EUR) ===

2019:
   1. EP14   1.97 Mrd  Bundesministerium der Verteidigung
   2. EP 6   1.90 Mrd  Bundesministerium des Innern, für Bau und Heimat
   3. EP30   1.59 Mrd  Bundesministerium für Bildung und Forschung
   4. EP12   0.93 Mrd  Bundesministerium für Verkehr und digitale Infrastruktu
   5. EP 9   0.90 Mrd  Bundesministerium für Wirtschaft und Energie
   6. EP 8   0.78 Mrd  Bundesministerium der Finanzen
   7. EP 4   0.08 Mrd  Bundeskanzlerin und Bundeskanzleramt
   8. EP 5   0.06 Mrd  Auswärtiges Amt
   9. EP 7   0.06 Mrd  Bundesministerium der Justiz und für Verbraucherschutz

2024:
   1. EP12   4.12 Mrd  Bundesministerium für Digitales und Verkehr
   2. EP30   3.88 Mrd  Bundesministerium für Bildung und Forschung
   3. EP14   3.12 Mrd  Bundesministerium der Verteidigung
   4. EP 8   2.00 Mrd  Bundesministerium der Finanzen
   5. EP 6   1.98 Mrd  Bundesministerium des Innern und für Heimat
   6. EP 9   1.38 Mrd  Bundesministerium für Wir

**Befund.** Die Spitzengruppe wechselt deutlich: BMVg (EP 14) fällt von Rang 1 (2019) auf Rang 3 (2024). Der Etat wächst absolut, aber langsamer als bei anderen. BMDV (EP 12) springt von Rang 4 auf Rang 1 und vervielfacht sein Digital-Volumen. BMBF (EP 30) bleibt durchgehend in der Spitzengruppe, das BMI (EP 6) rutscht von Rang 2 auf Rang 5 ab.

[TODO Schreib-Person: Rangwechsel zwischen 2019 und 2024 als Absatz ausformulieren. Auffälligkeiten: BMVg fällt von Rang 1 auf Rang 3, BMDV steigt von Rang 4 auf Rang 1.]

Diese Verschiebung ist mehr als Zahlen-Rotation — sie spiegelt politische Schwerpunktsetzung wider (Breitbandausbau, DigitalPakt). Nächster Schritt: Wofür geben die Top-Ressorts ihr Digital-Geld jeweils aus — ähneln sich die Aufgaben oder nicht?

In [5]:
#| label: tbl-profile
print("\n=== Profile der Top-Ressorts 2024 (anteilige Allokation) ===")
alloc = kategorien_allokation(df, jahr=2024)
profile = alloc.pivot_table(
    index='einzelplan', columns='kategorie', values='volumen', fill_value=0,
)
profile = profile / 1e6  # in Mrd.
profile['Sigma'] = profile.sum(axis=1)
top7 = profile.sort_values('Sigma', ascending=False).head(7)

# In Prozentanteilen
pct = top7.drop(columns='Sigma').div(top7['Sigma'], axis=0).mul(100).round(1)
print("\nProfile (in % der Ressort-Digital-Ausgaben):")
print(pct.to_string())

# Dominantes Profil pro Ressort
print("\nTop-2-Kategorien pro Ressort:")
for ep, row in pct.iterrows():
    sorted_k = row.sort_values(ascending=False)
    vol = top7.loc[ep, 'Sigma']
    print(
        f"  EP{int(ep):>2} ({vol:.2f} Mrd): "
        f"{sorted_k.index[0]} {sorted_k.iloc[0]:.0f}% | "
        f"{sorted_k.index[1]} {sorted_k.iloc[1]:.0f}%"
    )


=== Profile der Top-Ressorts 2024 (anteilige Allokation) ===

Profile (in % der Ressort-Digital-Ausgaben):
kategorie   Bundeswehr  Forschung  Gesundheit  Infrastruktur  Kompetenzen  Kultur  Unteilbar  Verwaltung  Wirtschaft
einzelplan                                                                                                          
12                 0.0        8.5         0.0           84.5          0.0     0.0        0.7         5.9         0.4
30                 0.0       46.8         2.0            7.5         43.4     0.0        0.0         0.3         0.0
14               100.0        0.0         0.0            0.0          0.0     0.0        0.0         0.0         0.0
8                  0.0        0.0         0.0           25.2          0.0     0.3        0.0        74.5         0.0
6                  0.0        0.2         0.1           17.7          0.4     0.0        0.0        81.7         0.0
9                  0.0       36.8         0.0           11.3          0.0

**Befund.** Die Profile bestätigen die Erwartung in aller Deutlichkeit: BMDV ist überwiegend ein Infrastruktur-Ressort (Breitbandausbau, ETCS), BMVg vollständig Bundeswehr-IT, BMBF teilt sich fast hälftig zwischen Forschungsförderung und digitalen Kompetenzen (DigitalPakt), BMF und BMI sind die einzigen klassischen Verwaltungs-Ressorts (ITZBund, Zoll, BSI, Digitalfunk), BMWK mischt Wirtschafts- und Forschungsförderung. Sechs Häuser, sechs grundverschiedene digitale Aufgaben, keine zwei Profile gleichen sich.

Das ist der Kern der polyzentrischen These: Diese Profile folgen der Sachpolitik der jeweiligen Ressorts, nicht einer gemeinsamen „Digital"-Logik. Eine Bündelung würde nicht nur Etats, sondern Fachaufgaben verschieben, der Verkehrsminister kann den Breitbandausbau nicht an ein BMDS abgeben, ohne die Verantwortung für digitale Infrastruktur insgesamt aus der Hand zu geben.

Nächster Schritt: Wie viel Volumen bliebe realistisch bündelbar, wenn man diese Sachlogik respektiert?

In [6]:
#| label: tbl-szenarien
print("\n=== BMDS-Buendelungs-Szenarien fuer 2024 ===")
total = df[df['jahr'] == 2024]['digi_soll_eng'].sum() / 1e6
print(f"Digital-Soll 2024 gesamt: {total:.2f} Mrd EUR\n")

# Helper: anteiliges Volumen einer Auswahl
def vol_subset(df_year, ep_list=None, kat_list=None):
    sub = df_year.copy()
    if ep_list is not None:
        sub = sub[sub['einzelplan'].isin(ep_list)]
    sub['n_kat'] = sub[KATS].sum(axis=1)
    sub = sub[sub['n_kat'] >= 1]
    if kat_list is None:
        return sub['digi_soll_eng'].sum() / 1e6
    v = 0.0
    for k in kat_list:
        part = sub[sub[k] == 1]
        v += (part['digi_soll_eng'] / part['n_kat']).sum()
    return v / 1e6

d24 = df[df['jahr'] == 2024].copy()

# Szenarien
zivile_ep = [4, 5, 6, 7, 8, 15, 16, 17, 25, 30]  # ohne EP 12 BMDV, ohne EP 14 BMVg

szen = {
    'Minimal-BMDS (Verwaltungs-IT ziviler Ressorts ohne BMDV)':
        vol_subset(d24, ep_list=zivile_ep, kat_list=['_3_dig_verwaltung']),
    'Realistisch (zusaetzlich BMI-Infrastruktur)':
        (vol_subset(d24, ep_list=zivile_ep, kat_list=['_3_dig_verwaltung']) +
         vol_subset(d24, ep_list=[6], kat_list=['_1_infra'])),
    'Maximal (alles ausser Bundeswehr & Forschung)':
        vol_subset(
            d24,
            ep_list=[4, 5, 6, 7, 8, 9, 10, 12, 15, 16, 17, 25, 30],
            kat_list=[
                '_1_infra', '_2_dig_wirtschaft', '_3_dig_verwaltung',
                '_4_dig_kompetenzen', '_5_dig_kultur',
                '_7_gesundheitswesen', '_9_unteibare_ausg',
            ],
        ),
}
for name, v in szen.items():
    mrd_str = f"{v:.2f}".replace(".", ",")
    pct_str = f"{v/total*100:.1f}".replace(".", ",")
    print(f"  {name}: {mrd_str} Mrd. ({pct_str} % des Digital-Solls)")


=== BMDS-Buendelungs-Szenarien fuer 2024 ===
Digital-Soll 2024 gesamt: 17.94 Mrd EUR

  Minimal-BMDS (Verwaltungs-IT ziviler Ressorts ohne BMDV): 3,70 Mrd. (20,6 % des Digital-Solls)
  Realistisch (zusaetzlich BMI-Infrastruktur): 4,05 Mrd. (22,6 % des Digital-Solls)
  Maximal (alles ausser Bundeswehr & Forschung): 11,77 Mrd. (65,6 % des Digital-Solls)


**Befund.** Die drei Szenarien zeigen die Bandbreite: Ein Minimal-Szenario (Verwaltungs-IT der zivilen Ressorts) und ein realistisches Szenario (zusätzlich BMI-Infrastruktur) liegen beide in der Größenordnung von gut einem Fünftel des Digital-Solls, nah beieinander, weil das BMI-Infrastruktur-Volumen klein ist. Das Maximal-Szenario kommt zwar auf deutlich mehr, verlangt aber, dass BMDV, BMBF und BMWK zentrale Sachaufgaben abgeben. Politisch ist das nicht zu erwarten.

Damit ist die Kernfrage von H1 beantwortet: Wie viel des Digitalhaushalts könnte ein BMDS bündeln, ohne tief in die Sachpolitik der Fachressorts einzugreifen?

## Kernbefund und weiter

**Der Digitalhaushalt ist polyzentrisch auf sechs Ressorts mit grundverschiedenen Aufgaben verteilt, eine BMDS-Bündelung ist auf realistisch rund ein Fünftel des Volumens begrenzt, der Rest bleibt aus fachlichen Gründen bei den Ressorts.** Die ursprüngliche These von „Schatten-Digitalministerien", die heimlich mehr ausgeben als das zuständige Haus, hält der Prüfung nicht stand, die Realität ist komplexer und politisch interessanter: ein System aus sechs Häusern mit eigener Logik statt einer verborgenen Konzentration.

**Weiter zu Notebook 02** Akt 2: Was passiert mit dem Geld? Wenn die Steuerungshoheit ohnehin begrenzt ist, lohnt der Blick darauf, was der Bund mit seinem Anteil eigentlich tut: selbst aufbauen oder verteilen?

**Limitationen dieser Analyse.** Die Profile beruhen auf der anteiligen 1/n-Allokation bei Mehrfachkategorien (`kategorien_allokation()`), eine transparente, aber vereinfachende Setzung. Die Szenarien-Grenzen (z. B. „Verwaltungs-IT" vs. „Infrastruktur") sind eigene, nachvollziehbare Setzungen, keine amtlichen Kategorien; eine externe Validierung über die Geschäftsverteilungspläne der Top-Ressorts steht noch aus.